In [1]:
import pandas as pd
import numpy as np
from pathlib import Path
from collections import Counter

DATASET_PATH = Path(r"D:\Projects\Before-the-Breach\02_DATASET\raw\CICIDS2017")
CHUNK_SIZE = 100000

csv_files = sorted(DATASET_PATH.glob("*.csv"))

if not csv_files:
    raise FileNotFoundError(f"No CSV files found in {DATASET_PATH}")

print("=" * 90)
print("CIC-IDS2017 DATASET INSPECTION")
print("=" * 90)
print(f"\nNumber of CSV files found: {len(csv_files)}")

dataset_summaries = []
reference_columns = None

for file_path in csv_files:
    print("\n" + "=" * 90)
    print(f"FILE: {file_path.name}")
    print("=" * 90)

    header = pd.read_csv(file_path, nrows=0, encoding="utf-8-sig")
    columns = [str(col).strip() for col in header.columns]

    print("\nNumber of columns:", len(columns))
    print("\nColumn names:")
    print(columns)

    if reference_columns is None:
        reference_columns = columns
    else:
        missing_columns = sorted(set(reference_columns) - set(columns))
        extra_columns = sorted(set(columns) - set(reference_columns))
        print("\nColumn comparison with first file:")
        print("Missing columns:", missing_columns)
        print("Extra columns:", extra_columns)
        print("Same column structure:", columns == reference_columns)

    sample = pd.read_csv(
        file_path,
        nrows=5,
        encoding="utf-8-sig"
    )
    sample.columns = [str(col).strip() for col in sample.columns]

    print("\nFirst five records:")
    display(sample)

    print("\nData types from sample:")
    print(sample.dtypes)

    total_rows = 0
    missing_counts = Counter()
    label_counts = Counter()
    duplicate_count_within_chunks = 0
    infinity_counts = Counter()

    for chunk in pd.read_csv(
        file_path,
        chunksize=CHUNK_SIZE,
        encoding="utf-8-sig"
    ):
        chunk.columns = [str(col).strip() for col in chunk.columns]
        total_rows += len(chunk)

        missing_counts.update(chunk.isna().sum().to_dict())

        if "Label" in chunk.columns:
            label_counts.update(
                chunk["Label"].astype(str).str.strip().value_counts().to_dict()
            )

        duplicate_count_within_chunks += int(chunk.duplicated().sum())

        numeric_chunk = chunk.select_dtypes(include=np.number)
        if not numeric_chunk.empty:
            infinity_counts.update(
                np.isinf(numeric_chunk).sum().to_dict()
            )

    print("\nDATASET SUMMARY")
    print("-" * 60)
    print("Total records:", f"{total_rows:,}")
    print("Total columns:", len(columns))
    print("File size (MB):", round(file_path.stat().st_size / (1024 ** 2), 2))

    print("\nMissing values:")
    missing_series = pd.Series(missing_counts).reindex(columns).fillna(0).astype(int)
    print(missing_series[missing_series > 0] if missing_series.sum() > 0 else "No missing values found.")

    print("\nLabel distribution:")
    if label_counts:
        label_series = pd.Series(label_counts).sort_values(ascending=False)
        label_percentages = (label_series / total_rows * 100).round(2)
        print(pd.DataFrame({
            "Count": label_series,
            "Percentage": label_percentages
        }))
    else:
        print("Label column not found.")

    print("\nInfinite values:")
    infinity_series = pd.Series(infinity_counts).reindex(columns).fillna(0).astype(int)
    print(infinity_series[infinity_series > 0] if infinity_series.sum() > 0 else "No infinite values found.")

    print("\nDuplicate records within individual chunks:", f"{duplicate_count_within_chunks:,}")
    print("Note: This duplicate count checks within chunks, not across chunk boundaries.")

    dataset_summaries.append({
        "File": file_path.name,
        "Records": total_rows,
        "Columns": len(columns),
        "Size_MB": round(file_path.stat().st_size / (1024 ** 2), 2),
        "Labels": dict(label_counts),
        "Missing_Values": int(sum(missing_counts.values())),
        "Infinite_Values": int(sum(infinity_counts.values())),
        "Duplicates_Within_Chunks": duplicate_count_within_chunks,
        "Column_Structure_Matches": columns == reference_columns
    })

print("\n" + "=" * 90)
print("OVERALL CIC-IDS2017 SUMMARY")
print("=" * 90)

summary_df = pd.DataFrame(dataset_summaries)
display(summary_df)

print("\nTotal records across all files:", f"{summary_df['Records'].sum():,}")
print("Total dataset size (MB):", round(summary_df["Size_MB"].sum(), 2))
print("All files have identical column structure:", summary_df["Column_Structure_Matches"].all())

print("\nINSPECTION COMPLETED")

CIC-IDS2017 DATASET INSPECTION

Number of CSV files found: 8

FILE: Friday-WorkingHours-Afternoon-DDos.pcap_ISCX.csv

Number of columns: 79

Column names:
['Destination Port', 'Flow Duration', 'Total Fwd Packets', 'Total Backward Packets', 'Total Length of Fwd Packets', 'Total Length of Bwd Packets', 'Fwd Packet Length Max', 'Fwd Packet Length Min', 'Fwd Packet Length Mean', 'Fwd Packet Length Std', 'Bwd Packet Length Max', 'Bwd Packet Length Min', 'Bwd Packet Length Mean', 'Bwd Packet Length Std', 'Flow Bytes/s', 'Flow Packets/s', 'Flow IAT Mean', 'Flow IAT Std', 'Flow IAT Max', 'Flow IAT Min', 'Fwd IAT Total', 'Fwd IAT Mean', 'Fwd IAT Std', 'Fwd IAT Max', 'Fwd IAT Min', 'Bwd IAT Total', 'Bwd IAT Mean', 'Bwd IAT Std', 'Bwd IAT Max', 'Bwd IAT Min', 'Fwd PSH Flags', 'Bwd PSH Flags', 'Fwd URG Flags', 'Bwd URG Flags', 'Fwd Header Length', 'Bwd Header Length', 'Fwd Packets/s', 'Bwd Packets/s', 'Min Packet Length', 'Max Packet Length', 'Packet Length Mean', 'Packet Length Std', 'Packet Leng

,Destination Port,Flow Duration,Total Fwd Packets,Total Backward Packets,Total Length of Fwd Packets,Total Length of Bwd Packets,Fwd Packet Length Max,Fwd Packet Length Min,Fwd Packet Length Mean,Fwd Packet Length Std,...,min_seg_size_forward,Active Mean,Active Std,Active Max,Active Min,Idle Mean,Idle Std,Idle Max,Idle Min,Label
0,54865,3,2,0,12,0,6,6,6,0,...,20,0,0,0,0,0,0,0,0,BENIGN
1,55054,109,1,1,6,6,6,6,6,0,...,20,0,0,0,0,0,0,0,0,BENIGN
2,55055,52,1,1,6,6,6,6,6,0,...,20,0,0,0,0,0,0,0,0,BENIGN
3,46236,34,1,1,6,6,6,6,6,0,...,20,0,0,0,0,0,0,0,0,BENIGN
4,54863,3,2,0,12,0,6,6,6,0,...,20,0,0,0,0,0,0,0,0,BENIGN



Data types from sample:
Destination Port               int64
Flow Duration                  int64
Total Fwd Packets              int64
Total Backward Packets         int64
Total Length of Fwd Packets    int64
                               ...  
Idle Mean                      int64
Idle Std                       int64
Idle Max                       int64
Idle Min                       int64
Label                            str
Length: 79, dtype: object

DATASET SUMMARY
------------------------------------------------------------
Total records: 225,745
Total columns: 79
File size (MB): 73.55

Missing values:
Flow Bytes/s    4
dtype: int64

Label distribution:
         Count  Percentage
DDoS    128027       56.71
BENIGN   97718       43.29

Infinite values:
Flow Bytes/s      30
Flow Packets/s    34
dtype: int64

Duplicate records within individual chunks: 1,870
Note: This duplicate count checks within chunks, not across chunk boundaries.

FILE: Friday-WorkingHours-Afternoon-PortScan.pca

,Destination Port,Flow Duration,Total Fwd Packets,Total Backward Packets,Total Length of Fwd Packets,Total Length of Bwd Packets,Fwd Packet Length Max,Fwd Packet Length Min,Fwd Packet Length Mean,Fwd Packet Length Std,...,min_seg_size_forward,Active Mean,Active Std,Active Max,Active Min,Idle Mean,Idle Std,Idle Max,Idle Min,Label
0,22,1266342,41,44,2664,6954,456,0,64.975610,109.864573,...,32,0,0,0,0,0,0,0,0,BENIGN
1,22,1319353,41,44,2664,6954,456,0,64.975610,109.864573,...,32,0,0,0,0,0,0,0,0,BENIGN
2,22,160,1,1,0,0,0,0,0.000000,0.000000,...,32,0,0,0,0,0,0,0,0,BENIGN
3,22,1303488,41,42,2728,6634,456,0,66.536585,110.129945,...,32,0,0,0,0,0,0,0,0,BENIGN
4,35396,77,1,2,0,0,0,0,0.000000,0.000000,...,32,0,0,0,0,0,0,0,0,BENIGN



Data types from sample:
Destination Port               int64
Flow Duration                  int64
Total Fwd Packets              int64
Total Backward Packets         int64
Total Length of Fwd Packets    int64
                               ...  
Idle Mean                      int64
Idle Std                       int64
Idle Max                       int64
Idle Min                       int64
Label                            str
Length: 79, dtype: object

DATASET SUMMARY
------------------------------------------------------------
Total records: 286,467
Total columns: 79
File size (MB): 73.34

Missing values:
Flow Bytes/s    15
dtype: int64

Label distribution:
           Count  Percentage
PortScan  158930       55.48
BENIGN    127537       44.52

Infinite values:
Flow Bytes/s      356
Flow Packets/s    371
dtype: int64

Duplicate records within individual chunks: 47,775
Note: This duplicate count checks within chunks, not across chunk boundaries.

FILE: Friday-WorkingHours-Morning.pcap

,Destination Port,Flow Duration,Total Fwd Packets,Total Backward Packets,Total Length of Fwd Packets,Total Length of Bwd Packets,Fwd Packet Length Max,Fwd Packet Length Min,Fwd Packet Length Mean,Fwd Packet Length Std,...,min_seg_size_forward,Active Mean,Active Std,Active Max,Active Min,Idle Mean,Idle Std,Idle Max,Idle Min,Label
0,3268,112740690,32,16,6448,1152,403,0,201.5,204.724205,...,32,3.594286e+02,1.199802e+01,380,343,16100000,4.988048e+05,16400000,15400000,BENIGN
1,389,112740560,32,16,6448,5056,403,0,201.5,204.724205,...,32,3.202857e+02,1.574499e+01,330,285,16100000,4.987937e+05,16400000,15400000,BENIGN
2,0,113757377,545,0,0,0,0,0,0.0,0.000000,...,0,9.361829e+06,7.324646e+06,18900000,19,12200000,6.935824e+06,20800000,5504997,BENIGN
3,5355,100126,22,0,616,0,28,28,28.0,0.000000,...,32,0.000000e+00,0.000000e+00,0,0,0,0.000000e+00,0,0,BENIGN
4,0,54760,4,0,0,0,0,0,0.0,0.000000,...,0,0.000000e+00,0.000000e+00,0,0,0,0.000000e+00,0,0,BENIGN


... (9 outputs left out)

,Destination Port,Flow Duration,Total Fwd Packets,Total Backward Packets,Total Length of Fwd Packets,Total Length of Bwd Packets,Fwd Packet Length Max,Fwd Packet Length Min,Fwd Packet Length Mean,Fwd Packet Length Std,...,min_seg_size_forward,Active Mean,Active Std,Active Max,Active Min,Idle Mean,Idle Std,Idle Max,Idle Min,Label
0,80,38308,1,1,6,6,6,6,6.000000,0.000000,...,20,0,0,0,0,0,0,0,0,BENIGN
1,389,479,11,5,172,326,79,0,15.636364,31.449238,...,32,0,0,0,0,0,0,0,0,BENIGN
2,88,1095,10,6,3150,3150,1575,0,315.000000,632.561635,...,32,0,0,0,0,0,0,0,0,BENIGN
3,389,15206,17,12,3452,6660,1313,0,203.058823,425.778474,...,32,0,0,0,0,0,0,0,0,BENIGN
4,88,1092,9,6,3150,3152,1575,0,350.000000,694.509719,...,32,0,0,0,0,0,0,0,0,BENIGN



Data types from sample:
Destination Port               int64
Flow Duration                  int64
Total Fwd Packets              int64
Total Backward Packets         int64
Total Length of Fwd Packets    int64
                               ...  
Idle Mean                      int64
Idle Std                       int64
Idle Max                       int64
Idle Min                       int64
Label                            str
Length: 79, dtype: object

DATASET SUMMARY
------------------------------------------------------------
Total records: 692,703
Total columns: 79
File size (MB): 214.74

Missing values:
Flow Bytes/s    1008
dtype: int64

Label distribution:
                   Count  Percentage
BENIGN            440031       63.52
DoS Hulk          231073       33.36
DoS GoldenEye      10293        1.49
DoS slowloris       5796        0.84
DoS Slowhttptest    5499        0.79
Heartbleed            11        0.00

Infinite values:
Flow Bytes/s       289
Flow Packets/s    1297
dtype

,File,Records,Columns,Size_MB,Labels,Missing_Values,Infinite_Values,Duplicates_Within_Chunks,Column_Structure_Matches
0,Friday-WorkingHours-Afternoon-DDos.pcap_ISCX.csv,225745,79,73.55,"{'DDoS': 128027, 'BENIGN': 97718}",4,64,1870,True
1,Friday-WorkingHours-Afternoon-PortScan.pcap_IS...,286467,79,73.34,"{'BENIGN': 127537, 'PortScan': 158930}",15,727,47775,True
2,Friday-WorkingHours-Morning.pcap_ISCX.csv,191033,79,55.62,"{'BENIGN': 189067, 'Bot': 1966}",28,216,5404,True
3,Monday-WorkingHours.pcap_ISCX.csv,529918,79,168.73,{'BENIGN': 529918},64,810,17480,True
4,Thursday-WorkingHours-Afternoon-Infilteration....,288602,79,79.25,"{'BENIGN': 288566, 'Infiltration': 36}",18,396,28104,True
5,Thursday-WorkingHours-Morning-WebAttacks.pcap_...,170366,79,49.61,"{'BENIGN': 168186, 'Web Attack � Brute Force':...",20,250,4829,True
6,Tuesday-WorkingHours.pcap_ISCX.csv,445909,79,128.82,"{'BENIGN': 432074, 'FTP-Patator': 7938, 'SSH-P...",201,327,16647,True
7,Wednesday-workingHours.pcap_ISCX.csv,692703,79,214.74,"{'BENIGN': 440031, 'DoS Hulk': 231073, 'DoS sl...",1008,1586,71964,True



Total records across all files: 2,830,743
Total dataset size (MB): 843.66
All files have identical column structure: True

INSPECTION COMPLETED
